In [8]:
%pip install nbformat ipywidgets plotly


Note: you may need to restart the kernel to use updated packages.


In [9]:
import sys
import torch
import torchvision
import torchaudio

In [10]:
import ast
import json
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from time import time

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

import torch
import torchaudio
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset
import timm

from IPython.display import display, Markdown


In [11]:
ELICE_PROJECT_DIR = "/home/elicer"
ELICE_DATASET_DIR = "/mnt/elice/dataset"
PROJECT_DIR = ELICE_PROJECT_DIR if os.path.isdir(ELICE_PROJECT_DIR) else os.getcwd()
DEFAULT_DATA_PATH = os.path.join(ELICE_DATASET_DIR, "birdclef-2026") if os.path.isdir(ELICE_DATASET_DIR) else os.path.join(PROJECT_DIR, "birdclef-2026")

DATA_PATH = os.environ.get("BIRDCLEF_DATA_PATH", "/home/elicer/birdclef-2026")
BASELINE_HISTORY_CSV = "/home/elicer/history/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.csv" 
# fold1 ""
BASELINE_MODEL_PATH = "/home/elicer/models/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.pth"
#fold1 ""
BASELINE_PRED_CSV = "/home/elicer/preds_baseline_val.csv"

CANDIDATE_HISTORY_CSV = "/home/elicer/history/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.csv"
# "/home/elicer/history/00f3082d17793ffaba8519d9b2452b272551a1e56422f8a370645b352d9a453a.csv" #lower spegaug 
#"/home/elicer/history/ad9a9133ab8def11dc2a57fe0a078c29138b57b9adcba74f7fe6380f71aa3f10.csv"#wave 

#"/home/elicer/history/221c61258ef1f2eea1e99bcdfb55972e76a014398072ef2fd8a4a9ae7e868661.csv" #spegaug
 # #combine
 #"/home/elicer/history/4e99c64af4f555f6cd7172e44cfefb9493c00ff4fc0bc3ec66b2c528d33560f9.csv" #fold1
CANDIDATE_MODEL_PATH = "/home/elicer/models/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.pth"
#"/home/elicer/models/00f3082d17793ffaba8519d9b2452b272551a1e56422f8a370645b352d9a453a.pth"
#"/home/elicer/models/ad9a9133ab8def11dc2a57fe0a078c29138b57b9adcba74f7fe6380f71aa3f10.pth"
# "/home/elicer/models/221c61258ef1f2eea1e99bcdfb55972e76a014398072ef2fd8a4a9ae7e868661.pth"
#
#"/home/elicer/models/4e99c64af4f555f6cd7172e44cfefb9493c00ff4fc0bc3ec66b2c528d33560f9.pth"


CANDIDATE_PRED_CSV = "/home/elicer/preds_experimetn_val.csv"

BASELINE_NAME = os.environ.get("BIRDCLEF_BASELINE_NAME", "baseline")
CANDIDATE_NAME = os.environ.get("BIRDCLEF_CANDIDATE_NAME", "experiment")
FOLD = int(os.environ.get("BIRDCLEF_FOLD", "0"))
N_SPLITS = int(os.environ.get("BIRDCLEF_N_SPLITS", "5"))
SEED = int(os.environ.get("BIRDCLEF_SEED", "2"))
TOP_K = int(os.environ.get("BIRDCLEF_TOP_K", "5"))
SAVE_PRED_CSV = True
SAVE_ANALYSIS_DIR = os.environ.get("BIRDCLEF_ANALYSIS_DIR", "/home/elicer/shift_analysis_notebook")

print("DATA_PATH:", DATA_PATH)
print("BASELINE_HISTORY_CSV:", BASELINE_HISTORY_CSV)
print("BASELINE_MODEL_PATH:", BASELINE_MODEL_PATH)
print("CANDIDATE_HISTORY_CSV:", CANDIDATE_HISTORY_CSV)
print("CANDIDATE_MODEL_PATH:", CANDIDATE_MODEL_PATH)
print("FOLD:", FOLD, "N_SPLITS:", N_SPLITS, "SEED:", SEED)


DATA_PATH: /home/elicer/birdclef-2026
BASELINE_HISTORY_CSV: /home/elicer/history/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.csv
BASELINE_MODEL_PATH: /home/elicer/models/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.pth
CANDIDATE_HISTORY_CSV: /home/elicer/history/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.csv
CANDIDATE_MODEL_PATH: /home/elicer/models/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.pth
FOLD: 0 N_SPLITS: 5 SEED: 2


In [12]:
def resolve_dataset_root(data_path=None):
    candidates = []
    if data_path:
        candidates.append(data_path)
    env_path = os.environ.get("BIRDCLEF_DATA_PATH")
    if env_path:
        candidates.append(env_path)
    candidates.extend([
        DEFAULT_DATA_PATH,
        ELICE_DATASET_DIR,
        os.path.join(ELICE_DATASET_DIR, "birdclef-2026"),
        os.path.join(PROJECT_DIR, "birdclef-2026"),
        PROJECT_DIR,
    ])
    for candidate in candidates:
        if not candidate:
            continue
        if os.path.exists(os.path.join(candidate, "taxonomy.csv")) and os.path.exists(os.path.join(candidate, "train_soundscapes_labels.csv")):
            return candidate
    raise FileNotFoundError("Could not find BirdCLEF dataset root. Checked: " + ", ".join([str(c) for c in candidates if c]))


def parse_label_tokens(value):
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    text = str(value).strip().strip("'\"")
    if text.lower() in {"", "nan", "none", "null"}:
        return []
    normalized = text.replace(";", ",").replace("|", ",")
    parts = normalized.split(",") if "," in normalized else normalized.split()
    tokens = []
    for part in parts:
        token = str(part).strip().strip("'\"")
        if token.lower() not in {"", "nan", "none", "null"}:
            tokens.append(token)
    return tokens


def unique_tokens(tokens):
    seen = set()
    ordered = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            ordered.append(token)
    return ordered


def first_existing_column(columns, candidates):
    lowered = {column.lower(): column for column in columns}
    for candidate in candidates:
        matched = lowered.get(candidate.lower())
        if matched is not None:
            return matched
    return None


def row_id_to_soundscape_id(row_id):
    row_id = str(row_id)
    if "_" not in row_id:
        return row_id
    return row_id.rsplit("_", 1)[0]


def parse_time_value(value):
    if value is None:
        return None
    if isinstance(value, (int, float, np.integer, np.floating)):
        if pd.isna(value):
            return None
        return float(value)
    text = str(value).strip()
    if text.lower() in {"", "nan", "none", "null"}:
        return None
    try:
        return float(text)
    except ValueError:
        pass
    try:
        return pd.to_timedelta(text).total_seconds()
    except ValueError:
        pass
    parts = text.split(":")
    if len(parts) >= 2:
        total = 0.0
        for part in parts:
            total = total * 60 + float(part)
        return total
    raise ValueError(f"Unsupported time format: {value}")


def format_window_suffix(seconds):
    rounded = int(round(seconds))
    if abs(seconds - rounded) < 1e-6:
        return str(rounded)
    return str(seconds).rstrip("0").rstrip(".")


def resolve_soundscape_filename(row, path_col, row_id_col):
    if path_col is not None:
        raw = str(row[path_col]).strip()
        if raw and raw.lower() != "nan":
            return os.path.basename(raw)
    if row_id_col is None:
        raise ValueError("Could not infer soundscape filename from labels.")
    soundscape_id = row_id_to_soundscape_id(row[row_id_col])
    if not soundscape_id.endswith(".ogg"):
        soundscape_id = f"{soundscape_id}.ogg"
    return soundscape_id


def load_soundscape_targets(data_path, label_columns):
    label_path = os.path.join(data_path, "train_soundscapes_labels.csv")
    labels_df = pd.read_csv(label_path)
    columns = list(labels_df.columns)
    row_id_col = first_existing_column(columns, ["row_id"])
    path_col = first_existing_column(columns, ["filename", "soundscape", "soundscape_filename", "file", "path", "audio_path"])
    label_col = first_existing_column(columns, ["primary_label", "label", "species", "target"])
    start_col = first_existing_column(columns, ["start", "start_sec", "start_time", "window_start", "begin"])
    end_col = first_existing_column(columns, ["end", "end_sec", "end_time", "window_end", "seconds", "stop"])

    aggregated = {}
    for row in labels_df.to_dict("records"):
        filename = resolve_soundscape_filename(row, path_col=path_col, row_id_col=row_id_col)
        if row_id_col is not None:
            row_id = str(row[row_id_col])
        else:
            end_sec = parse_time_value(row[end_col]) if end_col is not None else None
            start_sec = parse_time_value(row[start_col]) if start_col is not None else None
            if end_sec is None and start_sec is not None:
                end_sec = start_sec + 5.0
            if end_sec is None:
                raise ValueError("train_soundscapes_labels.csv must contain row_id or an end/start time column.")
            row_id = f"{os.path.splitext(filename)[0]}_{format_window_suffix(end_sec)}"
        entry = aggregated.setdefault(row_id, {"row_id": row_id, "filename": filename, "soundscape_id": os.path.splitext(filename)[0], "labels": []})
        if label_col is not None:
            entry["labels"].extend(parse_label_tokens(row[label_col]))

    records = []
    for entry in aggregated.values():
        target = np.zeros(len(label_columns), dtype=np.float32)
        for label in unique_tokens(entry["labels"]):
            if label in label_columns:
                target[label_columns.index(label)] = 1.0
        entry["target"] = target
        records.append(entry)

    target_df = pd.DataFrame([[record["row_id"], *record["target"].tolist()] for record in records], columns=["row_id", *label_columns])
    meta_df = pd.DataFrame(records)[["row_id", "filename", "soundscape_id"]]
    return target_df, meta_df


def select_validation_soundscapes(meta_df, fold, n_splits, seed):
    soundscape_ids = np.array(sorted(meta_df["soundscape_id"].unique()))
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(splitter.split(soundscape_ids))
    _, val_idx = splits[fold]
    val_ids = set(soundscape_ids[val_idx])
    val_meta = meta_df[meta_df["soundscape_id"].isin(val_ids)].copy()
    val_filenames = sorted(val_meta["filename"].unique().tolist())
    return val_meta, val_filenames


def macro_auc(targets, outputs):
    targets = (targets > 0).astype(float)
    scored_classes = np.sum(targets, axis=0) > 0
    return roc_auc_score(targets[:, scored_classes], outputs[:, scored_classes], average="macro")


In [13]:
class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512, f_min=20, f_max=16000, channels=1, norm="slaney", mel_scale="htk", target_size=(256, 256), top_db=80.0, delta_win=5,):
        super().__init__()
        self.channels = channels
        self.top_db = top_db
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=f_min,
            f_max=f_max,
            mel_scale=mel_scale,
            pad_mode="reflect",
            power=2.0,
            norm=norm,
            center=True,
        )
        print("channels", self.channels)
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S))
        if self.top_db is not None:
            max_val = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x, resize=True):
        squeeze = False
        if x.dim() == 1:
            x = x.unsqueeze(0)
            squeeze = True
        mel_spec = self.mel_transform(x)
        mel_spec = self.power_to_db(mel_spec)
        mel_spec = mel_spec.unsqueeze(1).repeat(1, self.channels, 1, 1)
        if resize:
            mel_spec = self.resize(mel_spec)
        B, C = mel_spec.shape[:2]
        flat = mel_spec.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel_spec = (mel_spec - mins) / (maxs - mins + 1e-7)
        if squeeze:
            mel_spec = mel_spec.squeeze(0)
        return mel_spec


class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = {
            "scale": 1,
            "backbone_pooling": "avg",
            "backbone": "tf_efficientnetv2_b0",
            "dropout": 0.1,
            "pretrained": True,
            "channels": 1,
            "num_labels": 234,
        }
        if config:
            self.config.update(config)
        self.training = True
        self.backbone = timm.create_model(
            self.config["backbone"],
            pretrained=self.config["pretrained"],
            num_classes=self.config["num_labels"],
            global_pool=self.config["backbone_pooling"],
            in_chans=1,
            drop_rate=self.config["dropout"],
        )

    def forward(self, x):
        labels = self.backbone(x)
        return labels


class BirdDataset(Dataset):
    def __init__(self, data_path, filenames):
        self.PATH = data_path
        self.TRAIN_PATH = os.path.join(data_path, "train_soundscapes")
        taxonomy = pd.read_csv(os.path.join(data_path, "taxonomy.csv"))
        self.LABELS = list(np.unique(taxonomy.primary_label))
        self.CLASSES = list(np.unique(taxonomy.class_name))
        self.BATCH_SIZE = 32
        self.DUR = 5
        self.SR = 32000
        self.paths = [os.path.join(self.TRAIN_PATH, filename) for filename in filenames]

    def __len__(self):
        return len(self.paths)


def format_time(t):
    h = int(t / 3600)
    t -= h * 3600
    m = int(t / 60)
    t -= m * 60
    out = ""
    if h > 0:
        out += str(h) + ":"
    return out + str(m).zfill(2) + ":" + str(int(t)).zfill(2)


def decode_config(cfg):
    for X in cfg:
        try:
            cfg[X] = ast.literal_eval(cfg[X]) if isinstance(cfg[X], str) else cfg[X]
        except Exception:
            try:
                cfg[X] = eval(cfg[X])
            except Exception:
                None
    return cfg


def predict_checkpoint(history_csv, model_path, output_csv=None, show_samples=True):
    data_path = resolve_dataset_root(DATA_PATH)
    label_columns = list(np.unique(pd.read_csv(os.path.join(data_path, "taxonomy.csv")).primary_label))
    target_df, meta_df = load_soundscape_targets(data_path, label_columns)
    val_meta, val_filenames = select_validation_soundscapes(meta_df=meta_df, fold=FOLD, n_splits=N_SPLITS, seed=SEED)
    dataset = BirdDataset(data_path, val_filenames)

    print("dataset_path:", data_path)
    print("history_csv:", history_csv)
    print("model_path:", model_path)
    print("fold:", FOLD, "n_splits:", N_SPLITS, "seed:", SEED)
    print("num_validation_soundscapes:", len(val_filenames))

    history = pd.read_csv(history_csv)
    config = {x: history.iloc[0][x] for x in history.columns[7:]}
    cfg = decode_config(config)
    cfg.update({"pretrained": False})
    spec = Spectrogram(**cfg["mel"])
    model = BirdModel(config=cfg)
    try:
        state_dict = torch.load(model_path, weights_only=True, map_location=torch.device("cpu"))
    except TypeError:
        state_dict = torch.load(model_path, map_location=torch.device("cpu"))
    model.load_state_dict(state_dict)

    def predict(filepath):
        wav, sr = torchaudio.load(filepath)
        n_seg = int(60 / dataset.DUR)
        wav = wav.float()[:, : dataset.SR * 60]
        n_repeat = 1
        wav = wav.float().reshape((n_seg, dataset.SR * dataset.DUR))
        activation = nn.Sigmoid()
        PREDS = []
        with torch.no_grad():
            mel = torch.stack([spec(wav[i]) for i in range(len(wav))])
            for _ in range(n_repeat):
                PREDS.append(activation(model(mel).unsqueeze(0)))
        PREDS = torch.concat(PREDS)
        pred = torch.mean(PREDS, dim=0)
        names = [ID + "_" + t for ID, t in zip([filepath.split("/")[-1].split(".")[0]] * n_seg, (np.array(range(n_seg)) * dataset.DUR + dataset.DUR).astype(str))]
        pred = pred.numpy()
        return pred, names

    pred = []
    names = []
    model.eval()
    start = time()
    with ThreadPoolExecutor(max_workers=4) as executor:
        for idx, (p, n) in enumerate(executor.map(predict, dataset.paths), 1):
            pred.append(p)
            names.append(n)
            fps = idx / (time() - start)
            if idx % 16 == 0 or idx == len(dataset):
                print(np.round(100 * idx / len(dataset), 1), "%", format_time(time() - start), "  -  remaining:", format_time((len(dataset) - idx) / fps))
        
    pred = np.concatenate(pred, axis=0)
    model_label_columns = dataset.LABELS[: pred.shape[1]]
    if pred.shape[1] > len(dataset.LABELS):
        raise ValueError(f"Model outputs {pred.shape[1]} classes, but taxonomy has only {len(dataset.LABELS)} labels.")
    print("prediction_shape:", pred.shape, "assigned_labels:", len(model_label_columns), "taxonomy_labels:", len(dataset.LABELS))
    pred_df = pd.DataFrame(np.zeros((len(pred), len(dataset.LABELS))), columns=dataset.LABELS)
    pred_df[model_label_columns] = pred
    pred_df.insert(0, "row_id", np.concatenate(names, axis=0))
    if output_csv:
        pred_df.to_csv(output_csv, index=False)
        print("saved:", output_csv)

    if show_samples and len(dataset.paths) > 0:
        for i in [min(2, len(dataset.paths) - 1), min(5, len(dataset.paths) - 1)]:
            filepath = dataset.paths[i]
            wav, sr = torchaudio.load(filepath)
            mel = spec(wav, resize=False)[0, 0]
            fig = px.imshow(pred.T[:, i * 12 : (i + 1) * 12], color_continuous_scale="Oranges", aspect="auto", zmax=1)
            fig.update_yaxes(autorange=True)
            fig.show()
            fig = px.imshow(mel, aspect="auto")
            fig.show()

    return pred_df, target_df, meta_df, dataset


In [14]:
baseline_pred, target_df, meta_df, dataset = predict_checkpoint(BASELINE_HISTORY_CSV, BASELINE_MODEL_PATH, BASELINE_PRED_CSV if SAVE_PRED_CSV else None, show_samples=False)
candidate_pred, _, _, _ = predict_checkpoint(CANDIDATE_HISTORY_CSV, CANDIDATE_MODEL_PATH, CANDIDATE_PRED_CSV if SAVE_PRED_CSV else None, show_samples=False)

display(Markdown("## Prediction Head"))
display(baseline_pred.head())
display(candidate_pred.head())


dataset_path: /home/elicer/birdclef-2026
history_csv: /home/elicer/history/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.csv
model_path: /home/elicer/models/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.pth
fold: 0 n_splits: 5 seed: 2
num_validation_soundscapes: 14
channels 1
100.0 % 00:06   -  remaining: 00:00
prediction_shape: (168, 234) assigned_labels: 234 taxonomy_labels: 234


/tmp/ipykernel_103559/2229349657.py:181: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pred_df.insert(0, "row_id", np.concatenate(names, axis=0))


saved: /home/elicer/preds_baseline_val.csv
dataset_path: /home/elicer/birdclef-2026
history_csv: /home/elicer/history/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.csv
model_path: /home/elicer/models/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.pth
fold: 0 n_splits: 5 seed: 2
num_validation_soundscapes: 14
channels 1
100.0 % 00:06   -  remaining: 00:00
prediction_shape: (168, 234) assigned_labels: 234 taxonomy_labels: 234
saved: /home/elicer/preds_experimetn_val.csv


/tmp/ipykernel_103559/2229349657.py:181: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pred_df.insert(0, "row_id", np.concatenate(names, axis=0))


## Prediction Head

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.001188,0.000009,0.000108,1.101919e-08,0.000075,1.579476e-07,0.000008,0.000633,0.000021,...,0.000019,0.000309,0.000005,0.000338,3.856205e-07,0.000203,0.000016,0.000027,0.000254,0.000384
1,BC2026_Train_0001_S08_20250606_030007_10,0.000122,0.000002,0.000032,1.964944e-09,0.000003,4.265882e-08,0.000003,0.000135,0.000004,...,0.000010,0.000193,0.000001,0.000175,2.524561e-08,0.000085,0.000004,0.000007,0.000136,0.000233
2,BC2026_Train_0001_S08_20250606_030007_15,0.000200,0.000007,0.000102,5.282852e-09,0.000009,1.251153e-07,0.000008,0.000584,0.000010,...,0.000014,0.000178,0.000002,0.000238,4.036869e-08,0.000056,0.000012,0.000021,0.000244,0.000488
3,BC2026_Train_0001_S08_20250606_030007_20,0.001766,0.000005,0.000051,9.873752e-09,0.000232,1.042244e-07,0.000001,0.000239,0.000025,...,0.000012,0.000304,0.000006,0.000556,2.225049e-07,0.000094,0.000025,0.000021,0.000128,0.000556
4,BC2026_Train_0001_S08_20250606_030007_25,0.000492,0.000003,0.000031,8.164407e-09,0.000085,7.439468e-08,0.000003,0.000315,0.000009,...,0.000014,0.000350,0.000004,0.000346,1.155829e-07,0.000079,0.000017,0.000016,0.000081,0.000433


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.133957,0.000078,0.000233,3.319214e-07,0.011367,0.000016,0.000044,0.005673,0.000150,...,0.000048,0.002578,0.000117,0.001401,2.283758e-06,0.000380,0.000140,0.000502,0.000177,0.003131
1,BC2026_Train_0001_S08_20250606_030007_10,0.095938,0.000086,0.000305,5.103344e-07,0.006141,0.000025,0.000059,0.009323,0.000226,...,0.000085,0.003375,0.000104,0.001616,1.975633e-06,0.000265,0.000213,0.000465,0.000355,0.002610
2,BC2026_Train_0001_S08_20250606_030007_15,0.122240,0.000067,0.000276,2.256102e-07,0.008078,0.000014,0.000056,0.007669,0.000278,...,0.000049,0.001748,0.000096,0.000972,8.746710e-07,0.000192,0.000077,0.000361,0.000156,0.002783
3,BC2026_Train_0001_S08_20250606_030007_20,0.143973,0.000049,0.000184,1.981818e-07,0.010474,0.000010,0.000039,0.006174,0.000131,...,0.000050,0.002166,0.000068,0.000715,1.401266e-06,0.000242,0.000148,0.000364,0.000127,0.002063
4,BC2026_Train_0001_S08_20250606_030007_25,0.154533,0.000082,0.000223,5.308743e-07,0.008516,0.000026,0.000058,0.009825,0.000242,...,0.000077,0.003698,0.000086,0.001830,1.760536e-06,0.000219,0.000316,0.000796,0.000269,0.002205


In [15]:
def evaluate_predictions(pred_df, target_df, label_columns):
    merged = target_df.merge(pred_df, on="row_id", how="inner", suffixes=("_true", "_pred"))
    if len(merged) != len(target_df):
        missing = set(target_df["row_id"]) - set(pred_df["row_id"])
        raise ValueError(f"Missing predictions for {len(missing)} row_ids.")
    target_matrix = merged[[f"{label}_true" for label in label_columns]].to_numpy()
    pred_matrix = merged[[f"{label}_pred" for label in label_columns]].to_numpy()
    return float(macro_auc(target_matrix, pred_matrix)), int(len(merged))


def build_prefixed_prediction_frames(target_df, baseline_pred, candidate_pred, label_columns):
    baseline_prefixed = baseline_pred.rename(columns={label: f"baseline__{label}" for label in label_columns})
    candidate_prefixed = candidate_pred.rename(columns={label: f"candidate__{label}" for label in label_columns})
    target_prefixed = target_df.rename(columns={label: f"target__{label}" for label in label_columns})
    merged = target_prefixed.merge(baseline_prefixed, on="row_id", how="inner")
    merged = merged.merge(candidate_prefixed, on="row_id", how="inner")
    return merged


def compute_overall_metrics(merged, label_columns):
    target_matrix = merged[[f"target__{label}" for label in label_columns]].to_numpy()
    baseline_matrix = merged[[f"baseline__{label}" for label in label_columns]].to_numpy()
    candidate_matrix = merged[[f"candidate__{label}" for label in label_columns]].to_numpy()
    pos_mask = target_matrix > 0
    neg_mask = target_matrix <= 0
    baseline_auc = float(macro_auc(target_matrix, baseline_matrix))
    candidate_auc = float(macro_auc(target_matrix, candidate_matrix))
    metrics = {
        "baseline_auc": baseline_auc,
        "candidate_auc": candidate_auc,
        "delta_auc": candidate_auc - baseline_auc,
        "baseline_positive_mean": float(baseline_matrix[pos_mask].mean()),
        "candidate_positive_mean": float(candidate_matrix[pos_mask].mean()),
        "delta_positive_mean": float(candidate_matrix[pos_mask].mean() - baseline_matrix[pos_mask].mean()),
        "baseline_negative_mean": float(baseline_matrix[neg_mask].mean()),
        "candidate_negative_mean": float(candidate_matrix[neg_mask].mean()),
        "delta_negative_mean": float(candidate_matrix[neg_mask].mean() - baseline_matrix[neg_mask].mean()),
        "baseline_separation": float(baseline_matrix[pos_mask].mean() - baseline_matrix[neg_mask].mean()),
        "candidate_separation": float(candidate_matrix[pos_mask].mean() - candidate_matrix[neg_mask].mean()),
    }
    metrics["delta_separation"] = metrics["candidate_separation"] - metrics["baseline_separation"]
    return metrics


def compute_per_class_metrics(merged, label_columns):
    records = []
    for label in label_columns:
        target = merged[f"target__{label}"].to_numpy()
        positive_count = int(target.sum())
        if positive_count == 0 or positive_count == len(target):
            continue
        baseline_scores = merged[f"baseline__{label}"].to_numpy()
        candidate_scores = merged[f"candidate__{label}"].to_numpy()
        baseline_auc = roc_auc_score(target, baseline_scores)
        candidate_auc = roc_auc_score(target, candidate_scores)
        pos_idx = target > 0
        neg_idx = target <= 0
        records.append({
            "label": label,
            "positive_count": positive_count,
            "baseline_auc": float(baseline_auc),
            "candidate_auc": float(candidate_auc),
            "delta_auc": float(candidate_auc - baseline_auc),
            "baseline_positive_mean": float(baseline_scores[pos_idx].mean()),
            "candidate_positive_mean": float(candidate_scores[pos_idx].mean()),
            "delta_positive_mean": float(candidate_scores[pos_idx].mean() - baseline_scores[pos_idx].mean()),
            "baseline_negative_mean": float(baseline_scores[neg_idx].mean()),
            "candidate_negative_mean": float(candidate_scores[neg_idx].mean()),
        })
    return pd.DataFrame(records).sort_values("delta_auc", ascending=False).reset_index(drop=True)


def build_row_level_metrics(merged, meta_df, label_columns):
    label_array = np.array(label_columns)
    meta_lookup = meta_df.set_index("row_id")
    row_records = []
    for _, row in merged.iterrows():
        target = row[[f"target__{label}" for label in label_columns]].to_numpy(dtype=float)
        baseline_scores = row[[f"baseline__{label}" for label in label_columns]].to_numpy(dtype=float)
        candidate_scores = row[[f"candidate__{label}" for label in label_columns]].to_numpy(dtype=float)
        pos_idx = target > 0
        neg_idx = ~pos_idx
        positive_labels = label_array[pos_idx].tolist()
        baseline_positive_mean = float(baseline_scores[pos_idx].mean()) if pos_idx.any() else float("nan")
        candidate_positive_mean = float(candidate_scores[pos_idx].mean()) if pos_idx.any() else float("nan")
        baseline_negative_mean = float(baseline_scores[neg_idx].mean()) if neg_idx.any() else float("nan")
        candidate_negative_mean = float(candidate_scores[neg_idx].mean()) if neg_idx.any() else float("nan")
        meta_row = meta_lookup.loc[row["row_id"]]
        row_records.append({
            "row_id": row["row_id"],
            "soundscape_id": meta_row["soundscape_id"],
            "filename": meta_row["filename"],
            "num_true_labels": int(pos_idx.sum()),
            "true_labels": ",".join(positive_labels),
            "baseline_positive_mean": baseline_positive_mean,
            "candidate_positive_mean": candidate_positive_mean,
            "delta_positive_mean": candidate_positive_mean - baseline_positive_mean,
            "baseline_negative_mean": baseline_negative_mean,
            "candidate_negative_mean": candidate_negative_mean,
            "baseline_separation": baseline_positive_mean - baseline_negative_mean,
            "candidate_separation": candidate_positive_mean - candidate_negative_mean,
            "delta_separation": (candidate_positive_mean - candidate_negative_mean) - (baseline_positive_mean - baseline_negative_mean),
        })
    return pd.DataFrame(row_records).sort_values("delta_positive_mean", ascending=False).reset_index(drop=True)


def build_soundscape_metrics(row_df):
    return (
        row_df.groupby("soundscape_id")
        .agg(
            filename=("filename", "first"),
            rows=("row_id", "count"),
            baseline_positive_mean=("baseline_positive_mean", "mean"),
            candidate_positive_mean=("candidate_positive_mean", "mean"),
            delta_positive_mean=("delta_positive_mean", "mean"),
            baseline_separation=("baseline_separation", "mean"),
            candidate_separation=("candidate_separation", "mean"),
            delta_separation=("delta_separation", "mean"),
        )
        .reset_index()
        .sort_values("delta_positive_mean", ascending=False)
        .reset_index(drop=True)
    )


In [16]:
data_path = resolve_dataset_root(DATA_PATH)
label_columns = list(np.unique(pd.read_csv(os.path.join(data_path, "taxonomy.csv")).primary_label))
target_df, meta_df = load_soundscape_targets(data_path, label_columns)
val_meta, val_filenames = select_validation_soundscapes(meta_df=meta_df, fold=FOLD, n_splits=N_SPLITS, seed=SEED)
val_row_ids = set(val_meta["row_id"])
target_df = target_df[target_df["row_id"].isin(val_row_ids)].reset_index(drop=True)
meta_df = meta_df[meta_df["row_id"].isin(val_row_ids)].reset_index(drop=True)

baseline_auc, _ = evaluate_predictions(baseline_pred, target_df, label_columns)
candidate_auc, _ = evaluate_predictions(candidate_pred, target_df, label_columns)
merged = build_prefixed_prediction_frames(target_df, baseline_pred, candidate_pred, label_columns)
overall_metrics = compute_overall_metrics(merged, label_columns)
per_class_df = compute_per_class_metrics(merged, label_columns)
row_df = build_row_level_metrics(merged, meta_df, label_columns)
soundscape_df = build_soundscape_metrics(row_df)

summary_df = pd.DataFrame([
    {"metric": "macro_auc", BASELINE_NAME: baseline_auc, CANDIDATE_NAME: candidate_auc, "delta": candidate_auc - baseline_auc},
    {"metric": "positive_mean", BASELINE_NAME: overall_metrics["baseline_positive_mean"], CANDIDATE_NAME: overall_metrics["candidate_positive_mean"], "delta": overall_metrics["delta_positive_mean"]},
    {"metric": "negative_mean", BASELINE_NAME: overall_metrics["baseline_negative_mean"], CANDIDATE_NAME: overall_metrics["candidate_negative_mean"], "delta": overall_metrics["delta_negative_mean"]},
    {"metric": "separation", BASELINE_NAME: overall_metrics["baseline_separation"], CANDIDATE_NAME: overall_metrics["candidate_separation"], "delta": overall_metrics["delta_separation"]},
])
display(Markdown(f"## Validation Summary\n- soundscapes: {len(val_filenames)}\n- rows: {len(target_df)}"))
display(summary_df)


## Validation Summary
- soundscapes: 14
- rows: 168

,metric,baseline,experiment,delta
0,macro_auc,0.758063,0.795577,0.037514
1,positive_mean,0.060357,0.072823,0.012466
2,negative_mean,0.001612,0.002726,0.001114
3,separation,0.058745,0.070097,0.011351


In [17]:
overall_fig = go.Figure()
overall_fig.add_trace(go.Bar(name=BASELINE_NAME, x=["macro AUC", "positive mean", "separation"], y=[overall_metrics["baseline_auc"], overall_metrics["baseline_positive_mean"], overall_metrics["baseline_separation"]]))
overall_fig.add_trace(go.Bar(name=CANDIDATE_NAME, x=["macro AUC", "positive mean", "separation"], y=[overall_metrics["candidate_auc"], overall_metrics["candidate_positive_mean"], overall_metrics["candidate_separation"]]))
overall_fig.update_layout(title="Overall soundscape robustness comparison", barmode="group", yaxis_title="score")
overall_fig.show()

hist_fig = px.histogram(per_class_df, x="delta_auc", nbins=40, title="Per-class AUC delta distribution")
hist_fig.add_vline(x=0.0, line_dash="dash", line_color="black")
hist_fig.show()

scatter_fig = px.scatter(per_class_df, x="baseline_auc", y="candidate_auc", size="positive_count", hover_name="label", title="Per-class AUC scatter")
min_val = float(min(per_class_df["baseline_auc"].min(), per_class_df["candidate_auc"].min()))
max_val = float(max(per_class_df["baseline_auc"].max(), per_class_df["candidate_auc"].max()))
scatter_fig.add_shape(type="line", x0=min_val, y0=min_val, x1=max_val, y1=max_val, line=dict(color="black", dash="dash"))
scatter_fig.show()


In [18]:
display(Markdown(f"## Top Improved Classes ({TOP_K})"))
display(per_class_df.head(TOP_K))
display(Markdown(f"## Top Degraded Classes ({TOP_K})"))
display(per_class_df.tail(TOP_K).sort_values("delta_auc"))
display(Markdown(f"## Top Improved Soundscapes ({TOP_K})"))
display(soundscape_df.head(TOP_K))
display(Markdown(f"## Top Degraded Soundscapes ({TOP_K})"))
display(soundscape_df.tail(TOP_K).sort_values("delta_positive_mean"))


## Top Improved Classes (5)

,label,positive_count,baseline_auc,candidate_auc,delta_auc,baseline_positive_mean,candidate_positive_mean,delta_positive_mean,baseline_negative_mean,candidate_negative_mean
0,limpki,1,0.586826,0.928144,0.341317,4.758854e-04,3.325293e-03,2.849408e-03,5.344434e-04,1.185151e-03
1,47158son05,1,0.670659,0.874251,0.203593,3.648239e-08,1.004397e-06,9.679150e-07,7.885119e-08,4.264192e-07
2,47158son17,12,0.443910,0.605235,0.161325,2.556759e-08,3.199053e-07,2.943377e-07,8.489362e-08,3.206148e-07
3,litnig1,6,0.525720,0.650206,0.124486,2.082623e-03,5.953959e-03,3.871335e-03,9.168595e-03,6.573468e-03
4,116570,3,0.785859,0.888889,0.103030,3.209748e-05,1.608162e-04,1.287188e-04,7.781970e-06,4.407667e-05


## Top Degraded Classes (5)

,label,positive_count,baseline_auc,candidate_auc,delta_auc,baseline_positive_mean,candidate_positive_mean,delta_positive_mean,baseline_negative_mean,candidate_negative_mean
32,47158son07,12,0.952457,0.827991,-0.124466,4.493320e-07,7.980188e-07,3.486868e-07,3.017595e-08,3.002944e-07
31,24279,33,0.965657,0.884400,-0.081257,1.427987e-01,1.645751e-01,2.177645e-02,1.227438e-03,2.879601e-02
30,trsowl,2,0.831325,0.774096,-0.057229,5.764269e-02,5.056388e-02,-7.078808e-03,8.801559e-03,2.446761e-02
29,47158son08,5,0.819632,0.766871,-0.052761,1.424423e-07,7.958855e-07,6.534432e-07,6.489339e-08,4.275703e-07
28,47158son01,12,0.927350,0.888355,-0.038996,3.228899e-07,9.233582e-07,6.004683e-07,7.097828e-08,3.381532e-07


## Top Improved Soundscapes (5)

,soundscape_id,filename,rows,baseline_positive_mean,candidate_positive_mean,delta_positive_mean,baseline_separation,candidate_separation,delta_separation
0,BC2026_Train_0053_S22_20220211_204500,BC2026_Train_0053_S22_20220211_204500.ogg,12,0.079645,0.184461,0.104816,0.079278,0.182441,0.103163
1,BC2026_Train_0047_S22_20220131_003000,BC2026_Train_0047_S22_20220131_003000.ogg,12,0.075823,0.172928,0.097105,0.075622,0.172081,0.096459
2,BC2026_Train_0033_S22_20211216_200000,BC2026_Train_0033_S22_20211216_200000.ogg,12,0.178873,0.214397,0.035523,0.177851,0.212715,0.034865
3,BC2026_Train_0052_S22_20220210_210000,BC2026_Train_0052_S22_20220210_210000.ogg,12,0.075043,0.101653,0.026611,0.074172,0.100225,0.026052
4,BC2026_Train_0031_S22_20211214_204500,BC2026_Train_0031_S22_20211214_204500.ogg,12,0.030399,0.043628,0.013229,0.030200,0.040814,0.010614


## Top Degraded Soundscapes (5)

,soundscape_id,filename,rows,baseline_positive_mean,candidate_positive_mean,delta_positive_mean,baseline_separation,candidate_separation,delta_separation
13,BC2026_Train_0049_S22_20220203_221500,BC2026_Train_0049_S22_20220203_221500.ogg,12,1.333977e-01,0.080055,-0.053343,0.131660,0.078375,-0.053285
12,BC2026_Train_0058_S15_20250617_060100,BC2026_Train_0058_S15_20250617_060100.ogg,12,1.376110e-01,0.085340,-0.052271,0.136095,0.083696,-0.052400
11,BC2026_Train_0037_S22_20211226_021500,BC2026_Train_0037_S22_20211226_021500.ogg,12,1.304077e-01,0.104946,-0.025462,0.129246,0.103221,-0.026025
10,BC2026_Train_0011_S03_20211114_200000,BC2026_Train_0011_S03_20211114_200000.ogg,12,6.008358e-03,0.001209,-0.004799,0.001672,-0.003737,-0.005410
9,BC2026_Train_0001_S08_20250606_030007,BC2026_Train_0001_S08_20250606_030007.ogg,12,1.972818e-07,0.000003,0.000002,-0.003209,-0.004129,-0.000920


In [19]:
def show_soundscape_heatmap(soundscape_id, title_prefix):
    subset_rows = row_df[row_df["soundscape_id"] == soundscape_id].copy()
    if subset_rows.empty:
        print("No rows for", soundscape_id)
        return
    subset_rows["window_end_sec"] = subset_rows["row_id"].str.rsplit("_", n=1).str[-1].astype(float)
    subset_rows = subset_rows.sort_values("window_end_sec")
    ordered_row_ids = subset_rows["row_id"].tolist()
    merged_subset = merged[merged["row_id"].isin(ordered_row_ids)].copy()
    merged_subset["window_end_sec"] = merged_subset["row_id"].str.rsplit("_", n=1).str[-1].astype(float)
    merged_subset = merged_subset.sort_values("window_end_sec")
    active_labels = [label for label in label_columns if merged_subset[f"target__{label}"].sum() > 0]
    if not active_labels:
        print("No active labels for", soundscape_id)
        return
    baseline_matrix = merged_subset[[f"baseline__{label}" for label in active_labels]].to_numpy().T
    candidate_matrix = merged_subset[[f"candidate__{label}" for label in active_labels]].to_numpy().T
    delta_matrix = candidate_matrix - baseline_matrix
    times = merged_subset["window_end_sec"].astype(int).astype(str).tolist()
    fig = make_subplots(rows=1, cols=3, subplot_titles=[BASELINE_NAME, CANDIDATE_NAME, "delta"], horizontal_spacing=0.08)
    fig.add_trace(go.Heatmap(z=baseline_matrix, x=times, y=active_labels, coloraxis="coloraxis"), row=1, col=1)
    fig.add_trace(go.Heatmap(z=candidate_matrix, x=times, y=active_labels, coloraxis="coloraxis"), row=1, col=2)
    fig.add_trace(go.Heatmap(z=delta_matrix, x=times, y=active_labels, coloraxis="coloraxis2"), row=1, col=3)
    fig.update_layout(title=f"{title_prefix}: {soundscape_id}", coloraxis=dict(colorscale="Oranges"), coloraxis2=dict(colorscale="RdBu", cmid=0.0), height=max(500, 30 * len(active_labels)))
    fig.show()


top_improved_soundscape = soundscape_df.iloc[0]["soundscape_id"]
top_degraded_soundscape = soundscape_df.iloc[-1]["soundscape_id"]
display(Markdown(f"## Most Improved Soundscape\n`{top_improved_soundscape}`"))
show_soundscape_heatmap(top_improved_soundscape, "Most improved soundscape")
display(Markdown(f"## Most Degraded Soundscape\n`{top_degraded_soundscape}`"))
show_soundscape_heatmap(top_degraded_soundscape, "Most degraded soundscape")


## Most Improved Soundscape
`BC2026_Train_0053_S22_20220211_204500`

## Most Degraded Soundscape
`BC2026_Train_0049_S22_20220203_221500`

In [20]:
output_dir = Path(SAVE_ANALYSIS_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
per_class_df.to_csv(output_dir / "per_class_metrics.csv", index=False)
row_df.to_csv(output_dir / "per_row_metrics.csv", index=False)
soundscape_df.to_csv(output_dir / "per_soundscape_metrics.csv", index=False)
summary = {
    "fold": FOLD,
    "n_splits": N_SPLITS,
    "seed": SEED,
    "baseline_name": BASELINE_NAME,
    "candidate_name": CANDIDATE_NAME,
    "num_validation_soundscapes": len(val_filenames),
    "num_validation_rows": len(target_df),
    "overall_metrics": overall_metrics,
}
with open(output_dir / "summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)
display(Markdown(f"Saved csv/json to `{output_dir}`"))


Saved csv/json to `/home/elicer/shift_analysis_notebook`

In [21]:
# === Fold sweep configuration and helpers ===

FOLDS_TO_SUMMARIZE = [
    int(part.strip())
    for part in os.environ.get(
        "BIRDCLEF_FOLDS_TO_SUMMARIZE",
        ",".join(str(fold) for fold in range(N_SPLITS)),
    ).split(",")
    if part.strip()
]

BASELINE_HISTORY_TEMPLATE = os.environ.get("BIRDCLEF_BASELINE_HISTORY_TEMPLATE", "")
BASELINE_MODEL_TEMPLATE = os.environ.get("BIRDCLEF_BASELINE_MODEL_TEMPLATE", "")
BASELINE_PRED_TEMPLATE = os.environ.get("BIRDCLEF_BASELINE_PRED_TEMPLATE", "")

CANDIDATE_HISTORY_TEMPLATE = os.environ.get("BIRDCLEF_CANDIDATE_HISTORY_TEMPLATE", "")
CANDIDATE_MODEL_TEMPLATE = os.environ.get("BIRDCLEF_CANDIDATE_MODEL_TEMPLATE", "")
CANDIDATE_PRED_TEMPLATE = os.environ.get("BIRDCLEF_CANDIDATE_PRED_TEMPLATE", "")

# Fill these dicts instead of templates when fold filenames are irregular.
BASELINE_HISTORY_BY_FOLD = {}
BASELINE_MODEL_BY_FOLD = {}
BASELINE_PRED_BY_FOLD = {}

CANDIDATE_HISTORY_BY_FOLD = {}
CANDIDATE_MODEL_BY_FOLD = {}
CANDIDATE_PRED_BY_FOLD = {}

def resolve_fold_artifact(fold, artifact_by_fold, artifact_template, artifact_fallback, artifact_name):
    if fold in artifact_by_fold:
        return artifact_by_fold[fold]
    if artifact_template:
        return artifact_template.format(fold=fold)
    if artifact_fallback:
        return artifact_fallback
    raise ValueError(f"Missing {artifact_name} for fold {fold}.")


def resolve_fold_output_csv(fold, pred_by_fold, pred_template, pred_fallback):
    if fold in pred_by_fold:
        return pred_by_fold[fold]
    if pred_template:
        return pred_template.format(fold=fold)
    if not pred_fallback or not SAVE_PRED_CSV:
        return None
    stem, suffix = os.path.splitext(pred_fallback)
    return f"{stem}_fold{fold}{suffix}"


def predict_checkpoint_for_fold(history_csv, model_path, fold, n_splits, seed, output_csv=None, show_samples=False):
    data_path = resolve_dataset_root(DATA_PATH)
    label_columns = list(np.unique(pd.read_csv(os.path.join(data_path, "taxonomy.csv")).primary_label))
    full_target_df, full_meta_df = load_soundscape_targets(data_path, label_columns)
    val_meta, val_filenames = select_validation_soundscapes(
        meta_df=full_meta_df,
        fold=fold,
        n_splits=n_splits,
        seed=seed,
    )
    val_row_ids = set(val_meta["row_id"])
    target_df = full_target_df[full_target_df["row_id"].isin(val_row_ids)].reset_index(drop=True)
    meta_df = full_meta_df[full_meta_df["row_id"].isin(val_row_ids)].reset_index(drop=True)
    dataset = BirdDataset(data_path, val_filenames)

    print("dataset_path:", data_path)
    print("history_csv:", history_csv)
    print("model_path:", model_path)
    print("fold:", fold, "n_splits:", n_splits, "seed:", seed)
    print("num_validation_soundscapes:", len(val_filenames))

    history = pd.read_csv(history_csv)
    config = {x: history.iloc[0][x] for x in history.columns[7:]}
    cfg = decode_config(config)
    cfg.update({"pretrained": False})
    spec = Spectrogram(**cfg["mel"])
    model = BirdModel(config=cfg)
    try:
        state_dict = torch.load(model_path, weights_only=True, map_location=torch.device("cpu"))
    except TypeError:
        state_dict = torch.load(model_path, map_location=torch.device("cpu"))
    model.load_state_dict(state_dict)

    def predict(filepath):
        wav, sr = torchaudio.load(filepath)
        n_seg = int(60 / dataset.DUR)
        wav = wav.float()[:, : dataset.SR * 60]
        n_repeat = 1
        wav = wav.float().reshape((n_seg, dataset.SR * dataset.DUR))
        activation = nn.Sigmoid()
        preds = []
        with torch.no_grad():
            mel = torch.stack([spec(wav[i]) for i in range(len(wav))])
            for _ in range(n_repeat):
                preds.append(activation(model(mel).unsqueeze(0)))
        preds = torch.concat(preds)
        pred = torch.mean(preds, dim=0)
        names = [
            sample_id + "_" + t
            for sample_id, t in zip(
                [filepath.split("/")[-1].split(".")[0]] * n_seg,
                (np.array(range(n_seg)) * dataset.DUR + dataset.DUR).astype(str),
            )
        ]
        return pred.numpy(), names

    pred = []
    names = []
    model.eval()
    start = time()
    with ThreadPoolExecutor(max_workers=4) as executor:
        for idx, (p, n) in enumerate(executor.map(predict, dataset.paths), 1):
            pred.append(p)
            names.append(n)
            fps = idx / (time() - start)
            if idx % 16 == 0 or idx == len(dataset):
                print(
                    np.round(100 * idx / len(dataset), 1),
                    "%",
                    format_time(time() - start),
                    "  -  remaining:",
                    format_time((len(dataset) - idx) / fps),
                )

    pred = np.concatenate(pred, axis=0)
    model_label_columns = dataset.LABELS[: pred.shape[1]]
    if pred.shape[1] > len(dataset.LABELS):
        raise ValueError(
            f"Model outputs {pred.shape[1]} classes, but taxonomy has only {len(dataset.LABELS)} labels."
        )
    pred_df = pd.DataFrame(np.zeros((len(pred), len(dataset.LABELS))), columns=dataset.LABELS)
    pred_df[model_label_columns] = pred
    pred_df.insert(0, "row_id", np.concatenate(names, axis=0))
    if output_csv:
        pred_df.to_csv(output_csv, index=False)
        print("saved:", output_csv)

    if show_samples and len(dataset.paths) > 0:
        for i in [min(2, len(dataset.paths) - 1), min(5, len(dataset.paths) - 1)]:
            filepath = dataset.paths[i]
            wav, sr = torchaudio.load(filepath)
            mel = spec(wav, resize=False)[0, 0]
            fig = px.imshow(pred.T[:, i * 12 : (i + 1) * 12], color_continuous_scale="Oranges", aspect="auto", zmax=1)
            fig.update_yaxes(autorange=True)
            fig.show()
            fig = px.imshow(mel, aspect="auto")
            fig.show()

    return pred_df, target_df, meta_df, val_filenames, label_columns


def evaluate_fold_pair(fold):
    baseline_history_csv = resolve_fold_artifact(
        fold, BASELINE_HISTORY_BY_FOLD, BASELINE_HISTORY_TEMPLATE, BASELINE_HISTORY_CSV, "baseline history_csv"
    )
    baseline_model_path = resolve_fold_artifact(
        fold, BASELINE_MODEL_BY_FOLD, BASELINE_MODEL_TEMPLATE, BASELINE_MODEL_PATH, "baseline model_path"
    )
    candidate_history_csv = resolve_fold_artifact(
        fold, CANDIDATE_HISTORY_BY_FOLD, CANDIDATE_HISTORY_TEMPLATE, CANDIDATE_HISTORY_CSV, "candidate history_csv"
    )
    candidate_model_path = resolve_fold_artifact(
        fold, CANDIDATE_MODEL_BY_FOLD, CANDIDATE_MODEL_TEMPLATE, CANDIDATE_MODEL_PATH, "candidate model_path"
    )
    baseline_output_csv = resolve_fold_output_csv(
        fold, BASELINE_PRED_BY_FOLD, BASELINE_PRED_TEMPLATE, BASELINE_PRED_CSV
    )
    candidate_output_csv = resolve_fold_output_csv(
        fold, CANDIDATE_PRED_BY_FOLD, CANDIDATE_PRED_TEMPLATE, CANDIDATE_PRED_CSV
    )

    baseline_pred, target_df, meta_df, val_filenames, label_columns = predict_checkpoint_for_fold(
        baseline_history_csv,
        baseline_model_path,
        fold=fold,
        n_splits=N_SPLITS,
        seed=SEED,
        output_csv=baseline_output_csv,
        show_samples=False,
    )
    candidate_pred, candidate_target_df, _, _, candidate_label_columns = predict_checkpoint_for_fold(
        candidate_history_csv,
        candidate_model_path,
        fold=fold,
        n_splits=N_SPLITS,
        seed=SEED,
        output_csv=candidate_output_csv,
        show_samples=False,
    )

    if label_columns != candidate_label_columns:
        raise ValueError("Baseline and candidate label columns do not match for fold evaluation.")
    if set(target_df["row_id"]) != set(candidate_target_df["row_id"]):
        raise ValueError("Baseline and candidate validation row_ids do not match for fold evaluation.")

    merged = build_prefixed_prediction_frames(target_df, baseline_pred, candidate_pred, label_columns)
    overall_metrics = compute_overall_metrics(merged, label_columns)

    return pd.DataFrame(
        [
            {
                "fold": fold,
                "metric": "macro_auc",
                BASELINE_NAME: overall_metrics["baseline_auc"],
                CANDIDATE_NAME: overall_metrics["candidate_auc"],
                "delta": overall_metrics["delta_auc"],
                "num_validation_soundscapes": len(val_filenames),
                "num_validation_rows": len(target_df),
            },
            {
                "fold": fold,
                "metric": "positive_mean",
                BASELINE_NAME: overall_metrics["baseline_positive_mean"],
                CANDIDATE_NAME: overall_metrics["candidate_positive_mean"],
                "delta": overall_metrics["delta_positive_mean"],
                "num_validation_soundscapes": len(val_filenames),
                "num_validation_rows": len(target_df),
            },
            {
                "fold": fold,
                "metric": "negative_mean",
                BASELINE_NAME: overall_metrics["baseline_negative_mean"],
                CANDIDATE_NAME: overall_metrics["candidate_negative_mean"],
                "delta": overall_metrics["delta_negative_mean"],
                "num_validation_soundscapes": len(val_filenames),
                "num_validation_rows": len(target_df),
            },
            {
                "fold": fold,
                "metric": "separation",
                BASELINE_NAME: overall_metrics["baseline_separation"],
                CANDIDATE_NAME: overall_metrics["candidate_separation"],
                "delta": overall_metrics["delta_separation"],
                "num_validation_soundscapes": len(val_filenames),
                "num_validation_rows": len(target_df),
            },
        ]
    )


def sample_std(values):
    values = np.asarray(values, dtype=float)
    return float(np.std(values, ddof=1)) if len(values) > 1 else float("nan")


def build_fold_stat_summary(fold_metric_df):
    rows = []
    for metric, group in fold_metric_df.groupby("metric", sort=False):
        baseline_values = group[BASELINE_NAME].to_numpy(dtype=float)
        candidate_values = group[CANDIDATE_NAME].to_numpy(dtype=float)
        delta_values = group["delta"].to_numpy(dtype=float)

        baseline_mean = float(baseline_values.mean())
        candidate_mean = float(candidate_values.mean())
        delta_mean = float(delta_values.mean())

        baseline_std = sample_std(baseline_values)
        candidate_std = sample_std(candidate_values)
        delta_std = sample_std(delta_values)

        rows.append(
            {
                "metric": metric,
                f"{BASELINE_NAME}_mean": baseline_mean,
                f"{BASELINE_NAME}_std": baseline_std,
                f"{BASELINE_NAME}_mean_std": f"{baseline_mean:.6f} +/- {baseline_std:.6f}",
                f"{CANDIDATE_NAME}_mean": candidate_mean,
                f"{CANDIDATE_NAME}_std": candidate_std,
                f"{CANDIDATE_NAME}_mean_std": f"{candidate_mean:.6f} +/- {candidate_std:.6f}",
                "delta_mean": delta_mean,
                "delta_std": delta_std,
                "delta_mean_std": f"{delta_mean:.6f} +/- {delta_std:.6f}",
            }
        )
    return pd.DataFrame(rows)


In [22]:
# === Multi-fold mean/std evaluation ===

fold_artifacts_ready = bool(BASELINE_MODEL_TEMPLATE or BASELINE_MODEL_BY_FOLD) and bool(
    CANDIDATE_MODEL_TEMPLATE or CANDIDATE_MODEL_BY_FOLD
)

if not FOLDS_TO_SUMMARIZE:
    raise ValueError("FOLDS_TO_SUMMARIZE is empty.")

if not fold_artifacts_ready:
    display(
        Markdown(
            "## Fold Mean/Std\n"
            "Set either the `*_MODEL_TEMPLATE` values with a `{fold}` placeholder or fill the `*_MODEL_BY_FOLD` dicts above, then rerun this cell.\n\n"
            f"- baseline example: `{BASELINE_MODEL_PATH}`\n"
            f"- candidate example: `{CANDIDATE_MODEL_PATH}`\n"
        )
    )
else:
    fold_metric_frames = []
    for fold in FOLDS_TO_SUMMARIZE:
        print(f"\n===== fold {fold} =====")
        fold_metric_frames.append(evaluate_fold_pair(fold))

    fold_metric_df = pd.concat(fold_metric_frames, ignore_index=True)
    fold_summary_df = build_fold_stat_summary(fold_metric_df)

    display(Markdown(f"## Fold Summary ({len(FOLDS_TO_SUMMARIZE)} folds)"))
    display(fold_summary_df)
    display(Markdown("## Per-Fold Metrics"))
    display(fold_metric_df.sort_values(["metric", "fold"]).reset_index(drop=True))


## Fold Mean/Std
Set either the `*_MODEL_TEMPLATE` values with a `{fold}` placeholder or fill the `*_MODEL_BY_FOLD` dicts above, then rerun this cell.

- baseline example: `/home/elicer/models/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.pth`
- candidate example: `/home/elicer/models/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.pth`


In [23]:
import numpy as np
import pandas as pd
import plotly.express as px

def build_score_distribution_df(pred_df, target_df, label_columns, model_name="baseline"):
    merged = target_df.merge(
        pred_df,
        on="row_id",
        how="inner",
        suffixes=("_true", "_pred")
    )

    y_true = merged[[f"{label}_true" for label in label_columns]].to_numpy()
    y_score = merged[[f"{label}_pred" for label in label_columns]].to_numpy()

    return pd.DataFrame({
        "score": y_score.reshape(-1),
        "target": np.where(y_true.reshape(-1) > 0, "positive label", "negative label"),
        "model": model_name,
    })

baseline_dist_df = build_score_distribution_df(
    baseline_pred,
    target_df,
    label_columns,
    model_name="baseline"
)

fig = px.histogram(
    baseline_dist_df,
    x="score",
    color="target",
    nbins=80,
    histnorm="probability density",
    barmode="overlay",
    opacity=0.55,
    title="Baseline prediction score distribution: positive vs negative labels",
    labels={
        "score": "predicted probability",
        "target": "ground-truth label type",
    },
)

fig.update_layout(
    xaxis_range=[0, 1],
    yaxis_title="density",
)

fig.show()


In [24]:
candidate_dist_df = build_score_distribution_df(
    candidate_pred,
    target_df,
    label_columns,
    model_name="plus"
)

compare_dist_df = pd.concat(
    [baseline_dist_df, candidate_dist_df],
    ignore_index=True
)

fig = px.histogram(
    compare_dist_df,
    x="score",
    color="target",
    facet_col="model",
    nbins=80,
    histnorm="probability density",
    barmode="overlay",
    opacity=0.55,
    title="Prediction score distribution: baseline vs plus",
    labels={
        "score": "predicted probability",
        "target": "ground-truth label type",
    },
)

fig.update_layout(
    xaxis_range=[0, 1],
    yaxis_title="density",
)

fig.show()




In [25]:
positive_only_df = compare_dist_df[compare_dist_df["target"] == "positive label"].copy()

fig = px.histogram(
    positive_only_df,
    x="score",
    color="model",
    nbins=50,
    histnorm="probability density",
    barmode="overlay",
    opacity=0.55,
    title="Positive-label prediction score distribution: baseline vs plus",
    labels={
        "score": "predicted probability for true positive labels",
        "model": "model",
    },
)

fig.update_layout(
    xaxis_range=[0, 1],
    yaxis_title="density",
)

fig.show()

summary = (
    compare_dist_df
    .query("target == 'positive label'")
    .groupby("model")["score"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

display(summary)



,model,count,mean,median,std
0,baseline,723,0.060357,0.000484,0.165250
1,plus,723,0.072823,0.001556,0.182041


In [26]:
fig = px.histogram(
    per_class_df,
    x="delta_auc",
    nbins=40,
    title="Per-class AUC change: plus - baseline",
    labels={
        "delta_auc": "AUC change per class",
    },
)

fig.add_vline(x=0.0, line_dash="dash", line_color="black")
fig.show()

delta_summary = pd.DataFrame({
    "num_improved_classes": [(per_class_df["delta_auc"] > 0).sum()],
    "num_degraded_classes": [(per_class_df["delta_auc"] < 0).sum()],
    "mean_delta_auc": [per_class_df["delta_auc"].mean()],
    "median_delta_auc": [per_class_df["delta_auc"].median()],
})

display(delta_summary)


,num_improved_classes,num_degraded_classes,mean_delta_auc,median_delta_auc
0,22,11,0.037514,0.02297


In [27]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import roc_auc_score

baseline_pred, target_df, meta_df, dataset = predict_checkpoint(
    BASELINE_HISTORY_CSV,
    BASELINE_MODEL_PATH,
    BASELINE_PRED_CSV if SAVE_PRED_CSV else None,
    show_samples=False
)

candidate_pred, _, _, _ = predict_checkpoint(
    CANDIDATE_HISTORY_CSV,
    CANDIDATE_MODEL_PATH,
    CANDIDATE_PRED_CSV if SAVE_PRED_CSV else None,
    show_samples=False
)


thresholds = [1, 2, 3, 5, 10]

robustness_rows = []

for min_count in thresholds:
    subset = per_class_df[per_class_df["positive_count"] >= min_count].copy()

    robustness_rows.append({
        "min_positive_count": min_count,
        "num_classes": len(subset),
        "num_improved_classes": int((subset["delta_auc"] > 0).sum()),
        "num_degraded_classes": int((subset["delta_auc"] < 0).sum()),
        "mean_delta_auc": subset["delta_auc"].mean(),
        "median_delta_auc": subset["delta_auc"].median(),
    })

robustness_df = pd.DataFrame(robustness_rows)
display(robustness_df)






dataset_path: /home/elicer/birdclef-2026
history_csv: /home/elicer/history/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.csv
model_path: /home/elicer/models/a7f2a764e78656812b62e16a767bbc84834b5e6a7d4839f12c44d5ae7fb4ff38.pth
fold: 0 n_splits: 5 seed: 2
num_validation_soundscapes: 14
channels 1
100.0 % 00:07   -  remaining: 00:00
prediction_shape: (168, 234) assigned_labels: 234 taxonomy_labels: 234


/tmp/ipykernel_103559/2229349657.py:181: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pred_df.insert(0, "row_id", np.concatenate(names, axis=0))


saved: /home/elicer/preds_baseline_val.csv
dataset_path: /home/elicer/birdclef-2026
history_csv: /home/elicer/history/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.csv
model_path: /home/elicer/models/643f54355845bd7476006a03d5ac95a83fcaa888a0d8efd3442245582a036c3b.pth
fold: 0 n_splits: 5 seed: 2
num_validation_soundscapes: 14
channels 1
100.0 % 00:05   -  remaining: 00:00
prediction_shape: (168, 234) assigned_labels: 234 taxonomy_labels: 234


/tmp/ipykernel_103559/2229349657.py:181: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pred_df.insert(0, "row_id", np.concatenate(names, axis=0))


saved: /home/elicer/preds_experimetn_val.csv


,min_positive_count,num_classes,num_improved_classes,num_degraded_classes,mean_delta_auc,median_delta_auc
0,1,33,22,11,0.037514,0.022970
1,2,30,19,11,0.020506,0.009155
2,3,27,18,9,0.022339,0.009340
3,5,26,17,9,0.019235,0.009155
4,10,21,14,7,0.016742,0.009340


In [28]:
fig = px.scatter(
    per_class_df,
    x="positive_count",
    y="delta_auc",
    hover_name="label",
    size="positive_count",
    title="Per-class AUC delta vs positive sample count",
    labels={
        "positive_count": "number of positive rows",
        "delta_auc": "AUC change: plus - baseline",
    },
)

fig.add_hline(y=0.0, line_dash="dash", line_color="black")
fig.update_xaxes(type="log")
fig.show()


Plus의 개선은 positive_count가 매우 작은 class의 영향도 포함하지만, 더 많은 positive sample을 가진 class에서도 개선이 관찰된다. 
따라서 augmentation 방향이 완전히 우연한 rare-class 효과라고 보기는 어렵다. 
다만 class별 sample 수에 따른 변동성이 존재하므로, multi-fold 검증으로 안정성을 확인할 필요가 있다.

positive_count >= 5에서도 mean/median delta가 양수:
개선이 rare class만의 효과는 아님

positive_count >= 5에서 mean은 양수지만 median이 0 근처:
일부 class 개선폭이 평균을 끌어올림

positive_count >= 5에서 mean/median이 음수:
현재 개선은 rare class 영향이 큼
